# v26 — 민정 v6의 동반 변이·경로·핫스팟 + XGBoost/LR 차용

제공된 팀원 코드의 구조만 참고하며 보고된 OOF/Public은 재현된 성능으로 쓰지 않습니다. 그룹 3-fold, fold 내부 빈도 선별, inner 그룹 분할 early stopping 후 재학습, 고정 0.5/0.5 혼합으로 검증합니다. 형제 라벨 반전·COAD prior 보정·OOF 비중 탐색은 사용하지 않습니다. 현재/옛 종결 표기 정규화는 v25와 같습니다. 모델·코드·데이터 원본은 보존하고 test/sample은 마지막 고정 모델 추론에서만 사용합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts23=[]
def forbid_inference23(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts23.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v23')
sys.addaudithook(forbid_inference23)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v26';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)

def normalize_token25(token):
 return re.sub(r'^(p\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',token)
def canonical_frame25(frame):
 def one(v):
  if not isinstance(v,str) or 'X' not in v:return v
  return ' '.join(normalize_token25(t) for t in v.split())
 return frame.map(one)
def legacy_frame25(frame,mixed=False):
 def one(v):
  if not isinstance(v,str) or '*' not in v:return v
  def conv(m):
   token=m[0]
   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token
   return token[:-1]+'X'
  return re.sub(r'(?<!\S)(?:p\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\*(?!\S)',conv,v)
 return frame.map(one)

X25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline
import gc
AA26="ACDEFGHIKLMNPQRSTVWY"
TYPES26=["missense","silent","nonsense","frameshift","splice","indel","other"]
CFG={"TMB_LOW":5,"TMB_HIGH":30}
gene_pos={g:i for i,g in enumerate(genes)}


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v26


In [2]:
# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.
AA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))
# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.
def parse_token16(raw):
 s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core;core=normalize_token25(core)
 if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)
 core=normalize_token25(core)
 key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'
 def effect(a,b):
  if a=='*' and b=='*':return 'stop_retained'
  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'
  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'
  if a not in AA16 or b not in AA16:return 'ambiguous_residue'
  return 'synonymous' if a==b else 'missense'
 m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)
 if m:
  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)
  if 'X' in [a,b]:flags.append('ambiguous_X')
 else:
  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)
  if m:
   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1
   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):
    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')
   else:kind='complex_range';flags.append('unresolved_range')
  elif 'fs' in core:
   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None
  elif 'delins' in core:kind='delins'
  elif 'del' in core:kind='deletion'
  elif 'ins' in core:kind='insertion'
  elif 'dup' in core:kind='duplication'
  elif 'ext' in core:kind='extension'
  elif core=='=':kind='no_change_unspecified'
  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')
  if pos is None:
   # Protein coordinate only: do not mistake fs termination distance for the affected residue.
   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None
 return (key,kind,pos,parts,tuple(flags),dist)

def parse_rows16(frame):
 cols=list(frame.columns);cache={};rows=[]
 for row in frame.itertuples(index=False,name=None):
  mutations=[];missing=[];raw_n=0;duplicates=0
  for gene,value in zip(cols,row):
   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue
   if str(value).strip()=='WT':continue
   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]
   for token in unique:
    if token not in cache:cache[token]=parse_token16(token)
    events.append(cache[token])
   mutations.append((gene,tuple(events)))
  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})
 return rows

FIXTURES16=[('R132H','missense',132),('R132R','synonymous',132),('R213*','stop_gained',213),('*363*','stop_retained',363),('*213Q','stop_lost',213),('K16Rfs*23','frameshift',16),('1_2FS>FS','synonymous_block',1),('483_484MP>IA','block_change',483),('10_12AAA>DD','complex_range',10),('490del','deletion',490),('p.(Arg123His)','missense',123),('R123X','stop_gained',123),('A391','incomplete',391),('FS','unparsed',None),('G12delinsAA','delins',12)]
for token,expected_kind16,expected_pos16 in FIXTURES16:
 result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)
assert parse_token16('K16Rfs*23')[5]==23
assert parse_token16('1_2FS>FS')[3]==((1,'F','F','synonymous'),(2,'S','S','synonymous'))
assert 'uncertain' in parse_token16('p.(Arg123His)')[4]
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[1]['missing']==('G',)
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[2]['duplicate_tokens']==1
print('P43 annotation safety fixtures passed:',len(FIXTURES16),flush=True)
# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.
PANELS16={
 'P25_breast':('TP53','PIK3CA','GATA3','MAP3K1'),
 'P26_colorectal':('APC','TP53','SMAD4','PIK3CA','KRAS','ARID1A','SOX9','FAM123B','POLE','MLH1'),
 'P27_glioma':('IDH1','IDH2','TP53','ATRX','CIC','FUBP1','NOTCH1'),
 'P29_gbm':('EGFR','PDGFRA'),
 'P30_renal':('VHL','PBRM1','SETD2','ARID1A','SMARCA4'),
 'P31_papillary_renal':('MET','SETD2','FH'),
 'P32_ovarian':('TP53','NF1','BRCA1','BRCA2','RB1','CDK12'),
 'P33_uterine':('TP53','PTEN','CTNNB1','PIK3CA','ARID1A','KRAS','ARID5B','POLE'),
 'P34_thyroid':('BRAF','EIF1AX','PPM1D','CHEK2'),
 'P35_prostate':('SPOP','FOXA1','IDH1'),
 'P36_lung_ad':('RIT1','MGA','EGFR','RBM10','NF1','MET','ERBB2'),
 'P37_lung_scc':('TP53','HLA-A','NFE2L2','KEAP1','CDKN2A','RB1'),
 'P38_melanoma':('BRAF','NRAS','HRAS','KRAS','NF1','KIT'),
 'P39_gastric':('PIK3CA','RHOA'),
 'P40_headneck':('PIK3CA','TRAF3','TP53','CDKN2A','HRAS','CASP8','NOTCH1','NSD1','AJUBA','FAT1','NFE2L2')}
print('Literature panel subsets:',len(PANELS16),'(external biological prior; no external patient data)',flush=True)

assert parse_token16("p.Arg123X")==parse_token16("R123*")


P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)


In [3]:
GENES16=list(genes)
parsed17=parse_rows16(X)

parent17=np.arange(len(train))
def root17(i):
 while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]
 return i
def join17(a,b):parent17[root17(b)]=root17(a)
canonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row['mutations'])),tuple(sorted(row['missing'])))).encode()).hexdigest() for row in parsed17]
for identifiers in [groups,canonical17]:
 seen17={}
 for i,key in enumerate(identifiers):
  if key in seen17:join17(seen17[key],i)
  else:seen17[key]=i
G17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]
assert not set(G17[D17])&set(G17[hold])

SHA17=hashlib.sha256((DATA/"train.csv").read_bytes()).hexdigest()
print("Development groups",len(D17),flush=True)


Development groups 4960


In [4]:
# Adapted from supplied model_민정_v6.ipynb cells10/14. No teammate data or outputs executed.
PATHWAYS = {'p53': ['TP53', 'MDM2', 'MDM4', 'CDKN2A', 'ATM', 'CHEK1', 'CHEK2'], 'RTK_RAS': ['EGFR', 'ERBB2', 'ERBB3', 'ERBB4', 'FGFR1', 'FGFR2', 'FGFR3', 'FGFR4', 'MET', 'ALK', 'RET', 'KIT', 'PDGFRA', 'KRAS', 'NRAS', 'HRAS', 'BRAF', 'RAF1', 'NF1', 'MAP2K1', 'MAP2K2', 'SOS1', 'PTPN11'], 'PI3K': ['PIK3CA', 'PIK3CB', 'PIK3R1', 'PTEN', 'AKT1', 'AKT2', 'AKT3', 'MTOR', 'TSC1', 'TSC2', 'STK11', 'RICTOR', 'INPP4B'], 'Wnt': ['APC', 'CTNNB1', 'AXIN1', 'AXIN2', 'RNF43', 'ZNRF3', 'TCF7L2', 'GSK3B', 'AMER1'], 'CellCycle': ['RB1', 'CDK4', 'CDK6', 'CCND1', 'CCND2', 'CCND3', 'CCNE1', 'CDKN1A', 'CDKN1B', 'CDKN2B', 'E2F1', 'FBXW7'], 'Notch': ['NOTCH1', 'NOTCH2', 'NOTCH3', 'NOTCH4', 'FBXW7', 'SPEN'], 'Hippo': ['NF2', 'LATS1', 'LATS2', 'SAV1', 'YAP1', 'WWTR1', 'TEAD1'], 'TGFb': ['TGFBR1', 'TGFBR2', 'SMAD2', 'SMAD3', 'SMAD4', 'ACVR1B', 'ACVR2A'], 'NRF2': ['NFE2L2', 'KEAP1', 'CUL3'], 'SWI_SNF': ['ARID1A', 'ARID1B', 'ARID2', 'SMARCA4', 'SMARCB1', 'PBRM1', 'SMARCA2'], 'Chromatin': ['KMT2A', 'KMT2B', 'KMT2C', 'KMT2D', 'EZH2', 'SETD2', 'KDM6A', 'KDM5C', 'CREBBP', 'EP300', 'BAP1', 'ASXL1', 'DNMT3A', 'IDH1', 'IDH2', 'ATRX'], 'DDR_MMR': ['BRCA1', 'BRCA2', 'PALB2', 'ATR', 'POLE', 'POLD1', 'MLH1', 'MSH2', 'MSH6', 'PMS2', 'RAD51', 'RAD50', 'MRE11', 'FANCA', 'FANCD2', 'ERCC2'], 'Splicing': ['SF3B1', 'SRSF2', 'U2AF1', 'RBM10'], 'LongGenes': ['TTN', 'MUC16', 'MUC4', 'SYNE1', 'OBSCN', 'RYR2', 'RYR1', 'CSMD1', 'CSMD3', 'LRP1B', 'USH2A', 'FLG', 'PCLO']}

def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [5]:
TRUNC26={'stop_gained','frameshift'}
INDEL26={'deletion','insertion','delins','duplication'}
class Features26:
 def __init__(self,combos=True):self.combos=combos
 def extract(self,rows):
  gp={g:i for i,g in enumerate(genes)};mut=np.zeros((len(rows),len(genes)),bool);tc=np.zeros((len(rows),7),np.float32);ac=np.zeros((len(rows),400),np.float32);rr=[];tt=[]
  for r,row in enumerate(rows):
   for g,events26 in row['mutations']:
    mut[r,gp[g]]=True
    for key,kind,pos,parts,flags,dist in events26:
     typ='nonsense' if kind=='stop_gained' else ('silent' if kind in {'synonymous','synonymous_block'} else ('indel' if kind in INDEL26 else (kind if kind in {'missense','frameshift'} else 'other')))
     tc[r,TYPES26.index(typ)]+=1
     if kind=='missense' and parts:
      _,a,b,_=parts[0];rr.append(r);tt.append(f'{g}|{a}{pos}')
      if a in AA26 and b in AA26:ac[r,AA26.index(a)*20+AA26.index(b)]+=1
     elif kind in TRUNC26:rr.append(r);tt.append(f'{g}|trunc')
     elif kind in INDEL26:
      rr.extend([r,r]);tt.extend([f'{g}|indel',f'{g}|{pos}_indel'])
  return {'mut':mut,'type_cnt':tc,'aa_cnt':ac,'tok_rows':np.asarray(rr,dtype=int),'tok_names':tt}
 def fit(self,rows):
  ex=self.extract(rows);self.keep=np.flatnonzero(ex['mut'].sum(0)>=5);cnt=Counter(nm for _,nm in set(zip(ex['tok_rows'],ex['tok_names'])));self.tokens=sorted(k for k,v in cnt.items() if v>=5);self.tidx={k:i for i,k in enumerate(self.tokens)};self.aakeep=np.flatnonzero(ex['aa_cnt'].sum(0)>=20)
  self.names=self.compose(ex)[1];return self
 def compose(self,ex):
  mut=ex['mut'];tc=ex['type_cnt'];n=len(mut);ng=mut.sum(1);nev=tc.sum(1);blocks=[];names=[]
  def add(a,nm):blocks.append(np.asarray(a,np.float32).reshape(n,-1));names.extend(nm)
  add(mut[:,self.keep],[f'g_{genes[j]}' for j in self.keep]);add(np.c_[ng,np.log1p(ng),nev],['n_mut_genes','log_n_mut_genes','n_events']);add(tc,['cnt_'+s for s in TYPES26]);trunc=tc[:,[TYPES26.index('nonsense'),TYPES26.index('frameshift')]].sum(1)
  add(np.c_[tc[:,1]/(nev+1),tc[:,0]/(nev+1),trunc/(nev+1)],['ratio_silent','ratio_missense','ratio_trunc'])
  h=np.zeros((n,len(self.tokens)),np.float32)
  for r,t in zip(ex['tok_rows'],ex['tok_names']):
   j=self.tidx.get(t)
   if j is not None:h[r,j]=1
  add(h,['hs_'+s for s in self.tokens])
  if self.combos:
   c,cn=combo_features(ex);add(c,['cb_'+s for s in cn])
  if len(self.aakeep):add(ex['aa_cnt'][:,self.aakeep]/(tc[:,0]+1)[:,None],[f'aa_{AA26[j//20]}>{AA26[j%20]}' for j in self.aakeep])
  add(np.column_stack([mut[:,[gene_pos[g] for g in gs if g in gene_pos]].sum(1) for gs in PATHWAYS.values()]),['pw_'+k for k in PATHWAYS])
  return np.hstack(blocks),names
 def transform(self,rows):return self.compose(self.extract(rows))[0]

def new_xgb26(rounds,early=False):
 return XGBClassifier(objective='multi:softprob',n_estimators=rounds,learning_rate=.08,max_depth=5,subsample=.8,colsample_bytree=.3,tree_method='hist',eval_metric='mlogloss',random_state=26026,n_jobs=2,**({'early_stopping_rounds':50} if early else {}))
class Model26:
 def __init__(self,combos=True):self.combos=combos;self.classes_=labels.copy();self.genes_=list(genes)
 def fit(self,indices,rounds=None,seed=26026):
  if rounds is None:
   ia,ib=next(StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(indices)),yi[indices],G17[indices]));it,iv=indices[ia],indices[ib];assert not set(G17[it])&set(G17[iv]);innerpre=Features26(self.combos).fit([parsed17[i] for i in it]);a=innerpre.transform([parsed17[i] for i in it]);b=innerpre.transform([parsed17[i] for i in iv]);inner=new_xgb26(1200,True)
   print('V26 inner early stopping',self.combos,len(it),len(iv),len(innerpre.names),flush=True);inner.fit(a,yi[it],sample_weight=compute_sample_weight('balanced',yi[it]),eval_set=[(b,yi[iv])],verbose=False);rounds=int(inner.best_iteration)+1;del innerpre,inner,a,b;gc.collect()
  self.rounds_=int(rounds);self.pre=Features26(self.combos).fit([parsed17[i] for i in indices]);a=self.pre.transform([parsed17[i] for i in indices]);self.xgb=new_xgb26(self.rounds_)
  print('V26 outer/full refit',self.combos,len(indices),len(self.pre.names),self.rounds_,flush=True);self.xgb.fit(a,yi[indices],sample_weight=compute_sample_weight('balanced',yi[indices]),verbose=False)
  self.lr=make_pipeline(MaxAbsScaler(),LogisticRegression(C=.1,class_weight='balanced',max_iter=1000));self.lr.fit(a,yi[indices]);return self
 def predict_parts(self,rows):
  a=self.pre.transform(rows);return self.xgb.predict_proba(a),self.lr.predict_proba(a)
 def predict_rows(self,rows):
  a,b=self.predict_parts(rows);p=.5*a+.5*b;assert p.shape==(len(rows),len(self.classes_)) and np.isfinite(p).all() and np.allclose(p.sum(1),1);return p
 def predict_frame(self,frame):return self.predict_rows(parse_rows16(canonical_frame25(frame.loc[:,self.genes_])))


In [6]:
PROTOCOL26={'version':26,'source':'model_민정_v6.ipynb, supplied by user; code-only borrowing, teammate results not independently reproduced','main_candidate':'combos_on','ablation':'combos_off, explanatory only, not a recipe selection','xgb_lr_weights':[.5,.5],'gene_min_df':5,'hotspot_min_df':5,'aa_min_count':20,'xgb':{'depth':5,'lr':.08,'max_inner_rounds':1200,'subsample':.8,'colsample':.3},'lr_C':.1,'outer_seed':23033,'outer_folds':3,'inner_group_folds':5,'early_stopping':'inner train-only vocabulary + inner validation, then outer-training vocabulary refit and XGB round refit','design':'fixed before outcomes; no twin flip, no COAD prior, no OOF weight tuning','comparison':'v25 OOF aligned by exact train and validation indices','test_for_research':False,'historical_bias':'same repeatedly observed development patients; no external validation','normalization':'v25 legacy-X assumption retained, proposed after earlier test notation audit'}
(OUT/'protocol_v26.json').write_text(json.dumps(PROTOCOL26,indent=2));np.save(OUT/'classes.npy',labels)
oof26={k:np.zeros((len(D17),NC)) for k in ['combos_on','combos_off']};rows26=[];rounds26=[];f26=lambda t,p:float(f1_score(t,p.argmax(1),labels=np.arange(NC),average='macro',zero_division=0))
for fold,(aa,bb) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=23033).split(X.iloc[D17],yi[D17],G17[D17])):
 tr,va=D17[aa],D17[bb];assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold])
 # v25 caches only contain genuinely out-of-fold probabilities, never full-train predictions.
 reference_path26=WORK/f'artifacts/v25/notation_cv_fold{fold}.npz'
 expected_sha26='92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab'
 assert SHA17==expected_sha26,'Historical baseline is for the supplied training data only.'
 documented_v25_f1=[0.4813030372414948,0.4973466832302821,0.5006005643662187]
 if reference_path26.exists() and (WORK/'artifacts/v25/verification_v25.json').exists():
  with np.load(reference_path26) as q:
   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);reference=q['current'].copy()
  meta25=json.loads((WORK/'artifacts/v25/verification_v25.json').read_text());assert meta25['train_sha256']==SHA17
  baseline_f1_26=f26(yi[va],reference)
 else:
  # Published v25 fold scores, not a newly computed baseline prediction. Exact split fingerprints below.
  fingerprints26=['06c06464902eb31794a47a2ba66fbddb68356ca00f496ca48467bed9ce5c57e8', 'ee3e3488ac4ee01532dfd255f21afe072913d5016f76cb0f9809a3af3b86e7f0', 'b19d1be892ad77c9ef9b6f9712fc569695526bd7cce7b36f9b05eb2512286a0d']
  assert hashlib.sha256(tr.tobytes()+va.tobytes()).hexdigest()==fingerprints26[fold]
  baseline_f1_26=documented_v25_f1[fold]
 for combos in [False,True]:
  name='combos_on' if combos else 'combos_off';model26=Model26(combos).fit(tr,seed=26026+fold);p=model26.predict_rows([parsed17[i] for i in va]);oof26[name][bb]=p
  subset=X.iloc[va[:16]];assert np.allclose(model26.predict_frame(subset),model26.predict_frame(legacy_frame25(subset)),atol=1e-7)
  record={'fold':fold,'candidate':name,'features':len(model26.pre.names),'rounds':model26.rounds_,'macro_f1':f26(yi[va],p),'accuracy':float(accuracy_score(yi[va],p.argmax(1))),'v25_macro_f1':baseline_f1_26,'delta_vs_v25':f26(yi[va],p)-baseline_f1_26,'notation_probe_passed':True}
  rows26.append(record);pd.DataFrame(rows26).to_csv(OUT/'group_cv_results.csv',index=False);np.savez_compressed(OUT/f'fold{fold}_{name}.npz',tr=tr,va=va,probabilities=p,reference_macro_f1=baseline_f1_26);print('V26 GROUP RESULT',record,flush=True)
  if combos:rounds26.append(model26.rounds_)
  del model26;gc.collect()
table26=pd.DataFrame(rows26);summary26={'version':26,'results':table26.groupby('candidate')[['macro_f1','accuracy','delta_vs_v25']].mean().to_dict('index'),'main_candidate':'combos_on','formal_improvement_gate':'+0.003 mean F1 vs v25 and each fold >=-0.001','improvement_gate_passed':bool(table26[table26.candidate=='combos_on'].delta_vs_v25.mean()>=.003 and table26[table26.candidate=='combos_on'].delta_vs_v25.min()>=-.001),'model_saved':False,'public_score':None,'test_for_research':False,'train_sha256':SHA17}
np.savez_compressed(OUT/'oof_v26.npz',dev=D17,classes=labels,**oof26);(OUT/'verification_v26.json').write_text(json.dumps(summary26,indent=2))
# Save the predeclared main candidate as experimental even if comparison fails; do not relabel another model as v26.
final_rounds26=int(np.median(rounds26));print('V26 FULL TRAIN',final_rounds26,flush=True);final26=Model26(True).fit(np.arange(len(train)),rounds=final_rounds26);final26.training_data_sha256_=SHA17;final26.training_protocol_=PROTOCOL26
path26=OUT/'cancer_model_v26_minjung_features_experimental.pkl'
with path26.open('wb') as f:cloudpickle.dump(final26,f)
with path26.open('rb') as f:reload26=cloudpickle.load(f)
assert np.allclose(final26.predict_frame(X.iloc[:8]),reload26.predict_frame(legacy_frame25(X.iloc[:8])),atol=1e-7)
pd.DataFrame({'feature':final26.pre.names}).to_csv(OUT/'selected_variables.csv',index=False);assert not forbidden_attempts23
summary26.update(model_saved=True,model_sha256=hashlib.sha256(path26.read_bytes()).hexdigest(),final_rounds=final_rounds26,final_features=len(final26.pre.names),legacy_notation_reload_passed=True)
(OUT/'verification_v26.json').write_text(json.dumps(summary26,indent=2));print('V26 COMPLETE',json.dumps(summary26,indent=2),flush=True)


V26 inner early stopping False 2645 661 4574
V26 outer/full refit False 3306 4966 205
V26 GROUP RESULT {'fold': 0, 'candidate': 'combos_off', 'features': 4966, 'rounds': 205, 'macro_f1': 0.4704128757230708, 'accuracy': 0.4740024183796856, 'v25_macro_f1': 0.4813030372414948, 'delta_vs_v25': -0.010890161518424035, 'notation_probe_passed': True}
V26 inner early stopping True 2645 661 4597
V26 outer/full refit True 3306 4989 170
V26 GROUP RESULT {'fold': 0, 'candidate': 'combos_on', 'features': 4989, 'rounds': 170, 'macro_f1': 0.4698101742818638, 'accuracy': 0.47702539298669894, 'v25_macro_f1': 0.4813030372414948, 'delta_vs_v25': -0.011492862959631034, 'notation_probe_passed': True}
V26 inner early stopping False 2645 662 4717
V26 outer/full refit False 3307 5060 179
V26 GROUP RESULT {'fold': 1, 'candidate': 'combos_off', 'features': 5060, 'rounds': 179, 'macro_f1': 0.4746388199516597, 'accuracy': 0.4900181488203267, 'v25_macro_f1': 0.4973466832302821, 'delta_vs_v25': -0.022707863278622398

## 고정 실험 모델의 최종 제출 추론
개선 기준 미달일 때도 실험용 CSV를 별도로 보존합니다. 기준 모델보다 좋다고 자동 추천하지 않습니다. test는 이 단계의 predict에만 사용합니다.

In [7]:
# 별도 프로세스에서 고정 모델로 최종 test 예측만 수행합니다.
from pathlib import Path
import subprocess,sys
roots26=[Path.cwd(),*Path.cwd().parents]
repo26=next((p for p in roots26 if (p/'data/train.csv').is_file()),None)
if repo26 is None:raise FileNotFoundError('data 폴더가 필요합니다.')
work26=repo26/'models'/'혜진' if (repo26/'models'/'혜진').is_dir() else repo26
program26="import sys,os,json,hashlib,cloudpickle,numpy as np,pandas as pd\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\ndef guard(e,a):\n if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name=='train.csv':raise RuntimeError('Final inference does not read train.csv')\nsys.addaudithook(guard)\nmp,tp,sp,op=map(Path,sys.argv[1:5]);proof=json.loads((mp.parent/'verification_v26.json').read_text());assert proof['model_saved']\nassert hashlib.sha256(mp.read_bytes()).hexdigest()==proof['model_sha256']\nwith mp.open('rb') as f:model=cloudpickle.load(f)\nt=pd.read_csv(tp);s=pd.read_csv(sp);assert t.ID.is_unique and s.ID.is_unique and len(t)==len(s) and set(t.ID)==set(s.ID)\nx=t.loc[:,model.genes_];p=model.predict_frame(x);assert p.shape==(len(t),26) and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)\nidx=[10,3,500,0];assert np.allclose(model.predict_frame(x.iloc[idx]),p[idx],atol=1e-6)\ns.SUBCLASS=s.ID.map(pd.Series(model.classes_[p.argmax(1)],index=t.ID));assert not s.isna().any().any()\nop.parent.mkdir(parents=True,exist_ok=True)\npayload=s.to_csv(index=False).encode()\nif op.exists():assert op.read_bytes()==payload,'Existing v26 submission differs; preserve it.'\nelse:op.write_bytes(payload)\nassert pd.read_csv(op).ID.equals(s.ID)\nreport={'version':26,'rows':len(s),'model_sha256':proof['model_sha256'],'csv_sha256':hashlib.sha256(op.read_bytes()).hexdigest(),'phase':'final inference only','model_fitting':False,'test_used_for_validation':False,'public_score':None,'batch_invariance':True}\n(mp.parent/'submission_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps(report,ensure_ascii=False,indent=2))\n"
args26=[str(work26/'artifacts/v26/cancer_model_v26_minjung_features_experimental.pkl'),str(repo26/'data/test.csv'),str(repo26/'data/sample_submission.csv'),str(repo26/'submission/혜진_submission_v26_minjung_features_experimental.csv')]
run26=subprocess.run([sys.executable,'-c',program26,*args26],capture_output=True,text=True)
if run26.returncode:raise RuntimeError(run26.stderr)
print(run26.stdout)


{
  "version": 26,
  "rows": 2546,
  "model_sha256": "20e7af04f05834490657b817195aeffe892b3c1e29d58a5e5f046866ca5d63fc",
  "csv_sha256": "cc287e24938d14d88e7e1a5475504b7270f0a8b72d6392766af34a450918776f",
  "phase": "final inference only",
  "model_fitting": false,
  "test_used_for_validation": false,
  "public_score": null,
  "batch_invariance": true
}